# COVID-19 India Data Analysis 🇮🇳

This notebook demonstrates an end-to-end workflow for loading, cleaning, analysing and visualising COVID-19 data for India.


## 1. Imports and configuration

The notebook supports both a local CSV and the historical COVID-19 India raw-data API. Because public APIs can change or disappear, a local dataset is the most reproducible option.


In [ ]:
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 6)
DATA_PATH = '../data/covid_india.csv'
API_URL = 'https://api.covid19india.org/raw_data.json'


## 2. Load data


In [ ]:
def load_data():
    try:
        df = pd.read_csv(DATA_PATH)
        print(f'Loaded local dataset: {len(df):,} rows')
        return df
    except FileNotFoundError:
        print('Local CSV not found; attempting the historical API...')
        response = requests.get(API_URL, timeout=30)
        response.raise_for_status()
        payload = response.json()
        records = payload.get('raw_data', payload if isinstance(payload, list) else [])
        df = pd.DataFrame(records)
        print(f'Loaded API dataset: {len(df):,} rows')
        return df

# Uncomment to load data:
# df = load_data()
# df.head()


## 3. Cleaning template

COVID datasets use different column names. The following helper standardises names and detects common date/state/case fields.


In [ ]:
def standardise_columns(df):
    out = df.copy()
    out.columns = (out.columns.astype(str).str.strip().str.lower()
                   .str.replace(' ', '_', regex=False)
                   .str.replace('-', '_', regex=False))
    return out

# Example after loading:
# df = standardise_columns(df)
# print(df.columns.tolist())


## 4. Generic exploratory analysis

Run the following cells after loading a compatible dataset. Adjust the column names when necessary.


In [ ]:
# df.info()
# df.describe(include='all').T.head(20)
# df.isna().sum().sort_values(ascending=False).head(15)


In [ ]:
# Set these to the matching columns in your dataset.
# DATE_COL = 'date'
# STATE_COL = 'state'
# CONFIRMED_COL = 'confirmed'
# RECOVERED_COL = 'recovered'
# DEATHS_COL = 'deceased'

# analysis = df.copy()
# analysis[DATE_COL] = pd.to_datetime(analysis[DATE_COL], errors='coerce')
# analysis[CONFIRMED_COL] = pd.to_numeric(analysis[CONFIRMED_COL], errors='coerce').fillna(0)
# analysis = analysis.dropna(subset=[DATE_COL]).sort_values(DATE_COL)


## 5. Daily confirmed cases


In [ ]:
# daily = analysis.groupby(DATE_COL, as_index=False)[CONFIRMED_COL].sum()
# daily['daily_change'] = daily[CONFIRMED_COL].diff().fillna(0)
# daily.tail()

# ax = daily.plot(x=DATE_COL, y=CONFIRMED_COL, title='COVID-19 Confirmed Cases in India')
# ax.set_xlabel('Date')
# ax.set_ylabel('Confirmed cases')
# plt.tight_layout()
# plt.show()


## 6. State-wise analysis


In [ ]:
# state_totals = (analysis.groupby(STATE_COL, as_index=False)[CONFIRMED_COL]
#                  .sum().sort_values(CONFIRMED_COL, ascending=False))
# display(state_totals.head(10))

# top10 = state_totals.head(10).sort_values(CONFIRMED_COL)
# plt.barh(top10[STATE_COL], top10[CONFIRMED_COL])
# plt.title('Top 10 Indian States/Regions by Confirmed Cases')
# plt.xlabel('Confirmed cases')
# plt.tight_layout()
# plt.show()


## 7. Recovery and death trends

If the dataset contains recovered and deceased columns, aggregate them by date and compare their trends.


In [ ]:
# optional = [c for c in [RECOVERED_COL, DEATHS_COL] if c in analysis.columns]
# for col in optional:
#     analysis[col] = pd.to_numeric(analysis[col], errors='coerce').fillna(0)
#     trend = analysis.groupby(DATE_COL, as_index=False)[col].sum()
#     plt.plot(trend[DATE_COL], trend[col], label=col.title())
# plt.title('Recovery and Death Trends')
# plt.xlabel('Date')
# plt.ylabel('Count')
# plt.legend()
# plt.tight_layout()
# plt.show()


## 8. Findings to discuss

- Which states or regions recorded the largest case burden?
- During which periods did confirmed cases increase most rapidly?
- How did recoveries and deaths change over time?
- What limitations arise from reporting delays, missing values and changing definitions?

**Important:** This is an educational analysis and should not be used for medical or policy decisions.
